# Datamine COZONE Process Example

This notebook demonstrates how to configure and run the **`cozone`** process wrapper in `dmstudio`.

## Process Description

## COZONE

The process allows the user to investigate the relationship between mining parameters (bench height, minimum mining width, direction of mining, bench reference elevation) and the rocktype composition of each mining volume, or contact zone.

Contact zones are defined in terms of the constituent rocktypes; eg, zone 1 contains rocktypes 1, 3 and 7; and zone 2 rocktype 2 and 3, and zone 3 just rocktype 2.

The process requires as input a geological model with a numeric rocktype field, and calculates volumes categorized by contact zone. An optional output model may be produced, giving the contact code for each cell of the input model. The output contact zone model may be added to the original grade/rocktype model using process **[ADDMOD](<addmod.md>)** to create a combined model. (Note that the contact zone model must be first sorted on IJK before **ADDMOD** is used). This combined model may then be evaluated using process **[MODRES](<modres.md>)** and the results tabulated using process TABRES, to give grades and tonnages within user-defined perimeters for each contact zone category on each mining bench.

The starting point for contact zone modeling (the contact zone origin) is defined by the optional parameters @**CZXORIG** , @**CZYORIG** and @**CZZORIG**. All cells and sub-cells in the model with X, Y and Z co-ordinates greater than these values will then be classified by contact zone. Upper limits on cells and sub-cells may be imposed by using retrieval criteria on the XC, YC and ZC cell centre values (lower limits can also be imposed this way as well). Default values for these three parameters are the model origin coordinates.

The user defines a minimum mining width, a mining bench height, and a direction of mining. These are independent of the basic cell dimensions of the input rocktype model, and are defined using the parameters @**CZMWID** , @**CZBHHT** and @**DIRECTN**. The benches and mining widths must be parallel to the model axes.

Contact zones are defined using interactive prompting. Zones are numbered, starting at 1, with a maximum number of user definable zones of 18. For each zone the user must supply between 1 and 10 numeric rocktype codes. These are the rocktype codes which have been used in the input model. Although any numeric values are permitted, for the purpose of formatting it is best to restrict values to the range of -999.9 to 9999.9, using a maximum of one decimal place. Contact zones may be defined as any combination of rocktype codes; e.g.

## CONTACT ZONE 1 >1

## CONTACT ZONE 2 >1,2

## CONTACT ZONE 3 >1,4,8,2,99

## CONTACT ZONE 4 >8,1

However many zones the user has defined (N), a further two zones will always be added.

* N+1 - to include all other rocktype combinations.

* N+2 - to include material which has not been specifically modelled, i.e. volumes in the input model where there are no cells or sub-cells.

Volumes will be reported for all N+2 contact zones. However, cells or sub-cells which are in contact zone N+2 are not included in the output model.

There is a maximum of 15 different rocktypes which may be included in the definition of all contact zones. In the above example 5 rocktype codes are used (1, 2, 4, 8 and 99).

The process works through the input model starting at the contact zone origin, completing one bench before starting the next. Within a bench each mining slice is completed before the next is started, as illustrated in Figure 1. The next cell or sub-cell boundary is identified and a vertical plane is drawn from the top to the bottom of the bench perpendicular to the mining slice. The rocktype code for all material lying immediately ahead of this plane (or face) is identified and the material classified according to contact zone. The vertical plane is advanced and the process repeated until the contact zone changes. The rectangular volume enclosed by the two vertical planes, the bench top and bottom, and the slice sides is split into a cell and sub-cell structure consistent with the model parameters of the input model, and written to the output model file. The process is repeated for every slice and every bench.

An example of contact zone definition is shown in Figure 2. Although this is shown in two dimensions for ease of illustration, the rules are applied in all three dimensions.

[![](../Images_Commands/cozone1.gif)](<javascript:void\(0\);>)

_Mining slices within a bench_

[![](../Images_Commands/cozone2.gif)](<javascript:void\(0\);>)

Subcell structures in input and output models

### Input Files:

* **in** (Block Model):
  Model containing rock-type information in the form of a numeric rocktype code. The
  model must also contain at least the fields **XC, YC, ZC, XINC, YINC, ZINC, XMORIG,

## YMORIG, ZMORIG, NX, NY, NZ, IJK**.

  Required=Yes

### Output Files:

* **out** (Undefined):
  Output contact zone model. This will contain the standard 13 model fields plus a field
  named CONTZONE - the numeric contact zone code.
  Required=No

### Fields:

* **rockcode** (Any : IN):
  Name of field in input model file containing rocktype code.
  Default=Undefined
  Required=Yes

### Parameters:

* **czxorig**:
  X co-ordinate of start point for contact zone definition.Default=input model X origin
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **czyorig**:
  Y co-ordinate of start point for contact zone definition.Default=input model Y origin
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **czzorig**:
  Z co-ordinate of start point for contact zone definition.Default=input model Z origin
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **czmwid**:
  Mining width as measured along either X or Y axis direction of model, depending on the
  **DIRECTN** parameter. Default is the cell size in the direction parallel to the mining
  face. That is, **YINC** if **DIRECTN** ='X' , **XINC** if **DIRECTN** ='Y'
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **czbhht**:
  Bench height increment as measured vertically upwards from CZZORIG.Default is Z
  direction cell size [ZINC] of the input model
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **directn**:
  Direction of face advance [either 'X' or 'Y'] with a default of 'X'; this is
  perpendicular to the direction in which the mining width **CZMWID** is measured.
  Range=Undefined
  Values='X','Y'
  Default='X'
  Required=No

* **ijksort**:
  0 - do not sort output model IJK values 1 - sort output model on IJK values
  Range=0,1
  Values=0,1
  Default=1
  Required=No

* **print**:

* **Options** (0: for summary of input and total volumes by contact zone. =1 as 0 plus):
  volumes by bench (0)
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('cozone')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute cozone
print("Running cozone...")
dm_cmd.cozone(
    in_i='t_assays',  # required
    out_o='t_cozone_out',  # required
    rockcode_f='optional',  # required
    # czxorig_p='optional',  # optional
    # czyorig_p='optional',  # optional
    # czzorig_p='optional',  # optional
    # czmwid_p='optional',  # optional
    # czbhht_p='optional',  # optional
    # directn_p='X',  # optional
    # ijksort_p=1,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("cozone execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_cozone_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")